# Atias · Entrenar el detector de fruta

Entrena **RF-DETR** (licencia Apache 2.0, se puede usar en un producto comercial) para detectar fruta y su grado de madurez, y lo deja listo para el servicio de análisis de Atias.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*. Después, *Entorno de ejecución → Ejecutar todo*.

Tarda entre 1 y 3 horas con la GPU gratuita. Al final se descarga `modelo-<cultivo>.zip`: hay que descomprimirlo dentro de la carpeta `modelos/` de Atias. El análisis lo usa sin reiniciar nada.

| Paso | Qué hace |
|---|---|
| 1 | Configuración: cultivo, tamaño del modelo y datos propios |
| 2 | Instala las librerías |
| 3 | Prepara los datos: públicos y, si los hay, etiquetados de vuestros vídeos |
| 4 | Entrena |
| 5 | Mide el error de conteo en imágenes que el modelo no ha visto |
| 6 | Exporta a ONNX y comprueba que da lo mismo que el original |
| 7 | Empaqueta y descarga |

## 1. Configuración

- **`CULTIVO`**: `tomate_cherry`, `pimiento` o `pepino`.
- **Datos públicos**: para tomate se descargan solos (AgRobTomato, invernadero, CC BY 4.0). Para pimiento y pepino no hay un conjunto abierto que se descargue sin cuenta: hay que subir un ZIP en formato COCO (ver paso 3).
- **`DATOS_PROPIOS`**: ZIP en formato COCO con fotogramas etiquetados de vuestros vídeos. Con ellos el modelo mejora mucho. Déjalo en `None` la primera vez.
- **`MODO`**:
  - `fruto_color` (por defecto en tomate y pimiento): el modelo solo aprende a encontrar frutos, y Atias decide la madurez por el color. Es lo recomendable mientras falten fotos etiquetadas de fruta madura: en AgRobTomato hay 5.594 tomates verdes y solo 30 maduros.
  - `clases`: el modelo aprende también la madurez. Hace falta para el pepino (calibre) y conviene cuando haya fotogramas propios etiquetados.
- **`EQUIVALENCIAS`**: qué clases del modelo cuentan como maduras para Atias. Se pueden cambiar después en `modelo.json` sin volver a entrenar.

In [ ]:
CULTIVO = "tomate_cherry"          # tomate_cherry | pimiento | pepino
TAMANO = "medium"                  # nano (más rápido) | small | medium (recomendado) | large
EPOCAS = 40
DATOS_PROPIOS = None               # p. ej. "/content/propios.zip" (COCO). Súbelo con el icono de carpeta.
DATOS_PUBLICOS_ZIP = None          # Solo pimiento y pepino: ZIP COCO subido a mano.
MODO = "clases" if CULTIVO == "pepino" else "fruto_color"   # fruto_color | clases

# Clases del modelo y a qué clase de Atias equivalen (las de config/cultivos.plantilla.json).
CLASES = {
    "tomate_cherry": ["verde", "pinton", "rojizo", "maduro"],
    "pimiento": ["inmaduro", "maduro"],
    "pepino": ["pequeno", "comercial"],
}[CULTIVO]
EQUIVALENCIAS = {
    "tomate_cherry": {"verde": "inmaduro", "pinton": "inmaduro", "rojizo": "rojo", "maduro": "rojo"},
    "pimiento": {"inmaduro": "inmaduro", "maduro": "maduro"},
    "pepino": {"pequeno": "pequeno", "comercial": "comercial"},
}[CULTIVO]
MADURAS = {c for c, atias in EQUIVALENCIAS.items() if atias in ("rojo", "maduro", "comercial")}
if MODO == "fruto_color":
    assert CULTIVO != "pepino", "El pepino se clasifica por calibre: usa MODO = 'clases'"
    # Todas las clases se unen en «fruto»; null: Atias decide la madurez por el color.
    UNION = {c: "fruto" for c in CLASES}
    CLASES, EQUIVALENCIAS, MADURAS = ["fruto"], {"fruto": None}, {"fruto"}
else:
    UNION = {c: c for c in CLASES}

# Cómo lo usará el análisis: el fotograma (como mucho 1920 px de ancho) se parte en 2x2 teselas.
TESELAS, SOLAPE, ANCHO_MAX = [2, 2], 0.15, 1920
SEMILLA = 7

## 2. Instalar

In [ ]:
!pip -q install "rfdetr[train,onnx]==1.12.0" onnxruntime
import torch
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución → Cambiar tipo → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Preparar los datos

Todo se convierte a COCO con carpetas `train/`, `valid/` y `test/`, que es lo que espera RF-DETR. Las imágenes de prueba (`test/`) nunca se usan para entrenar: con ellas se mide el error.

**AgRobTomato** (tomate): INESC TEC, [Zenodo 5596799](https://zenodo.org/records/5596799), CC BY 4.0. 449 imágenes de invernadero con 4 grados de madurez.

In [ ]:
import json, random, shutil, zipfile, urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image

ROOT = Path("/content/atias")
SOURCES = ROOT / "fuentes"
DATASET = ROOT / "dataset"
shutil.rmtree(DATASET, ignore_errors=True)
SOURCES.mkdir(parents=True, exist_ok=True)
CATEGORIES = [{"id": i, "name": name, "supercategory": "fruto"} for i, name in enumerate(CLASES)]
LICENCIAS = []

def agrob_class(name):
    name = name.lower()
    if "unrip" in name: label = "verde"
    elif "break" in name: label = "pinton"
    elif "redd" in name: label = "rojizo"
    elif "rip" in name: label = "maduro"
    else: return None
    return UNION.get(label, label)

def find_image(xml, filename):
    if filename:
        found = next(iter(xml.parent.parent.rglob(filename)), None)
        if found: return found
    return next((xml.with_suffix(s) for s in (".jpg", ".png", ".jpeg") if xml.with_suffix(s).exists()), None)

def voc_to_items(folder):
    items, unknown = [], set()
    for xml in folder.rglob("*.xml"):
        root = ET.parse(xml).getroot()
        image = find_image(xml, root.findtext("filename"))
        if image is None:
            continue
        boxes = []
        for obj in root.iter("object"):
            label = agrob_class(obj.findtext("name") or "")
            if label is None:
                unknown.add(obj.findtext("name")); continue
            b = obj.find("bndbox")
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
            boxes.append((label, x1, y1, x2 - x1, y2 - y1))
        items.append((image, boxes))
    if unknown: print("Clases ignoradas:", unknown)
    return items

def coco_to_items(folder):
    items = []
    for ann_file in folder.rglob("*.json"):
        data = json.loads(ann_file.read_text())
        if "images" not in data: continue
        names = {c["id"]: c["name"] for c in data["categories"]}
        by_image = {}
        for a in data["annotations"]:
            by_image.setdefault(a["image_id"], []).append(a)
        for img in data["images"]:
            path = next(iter(ann_file.parent.rglob(Path(img["file_name"]).name)), None)
            if path is None: continue
            boxes = []
            for a in by_image.get(img["id"], []):
                name = UNION.get(names[a["category_id"]], names[a["category_id"]])
                if MODO == "fruto_color":
                    name = "fruto"
                if name not in CLASES:
                    raise ValueError(f"Clase {name!r} no está en CLASES={CLASES}")
                boxes.append((name, *a["bbox"]))
            items.append((path, boxes))
    return items

if CULTIVO == "tomate_cherry":
    zip_path = SOURCES / "agrob.zip"
    if not zip_path.exists():
        urllib.request.urlretrieve(
            "https://zenodo.org/records/5596799/files/Dataset-Greenhouse_Tomato_AgRob.zip?download=1", zip_path)
    zipfile.ZipFile(zip_path).extractall(SOURCES / "agrob")
    public = voc_to_items(SOURCES / "agrob")
    LICENCIAS.append("AgRobTomato Dataset (INESC TEC), https://zenodo.org/records/5596799, CC BY 4.0")
elif DATOS_PUBLICOS_ZIP:
    zipfile.ZipFile(DATOS_PUBLICOS_ZIP).extractall(SOURCES / "publicos")
    public = coco_to_items(SOURCES / "publicos")
    LICENCIAS.append("Datos públicos subidos a mano: revisar y anotar aquí su licencia")
else:
    public = []
own = []
if DATOS_PROPIOS:
    zipfile.ZipFile(DATOS_PROPIOS).extractall(SOURCES / "propios")
    own = coco_to_items(SOURCES / "propios")
    LICENCIAS.append("Fotogramas propios del piloto de Atias")
assert public or own, "No hay datos: revisa el paso 1"
print(f"Imágenes públicas: {len(public)} · propias: {len(own)}")

# Reparto: los datos propios pesan más en la prueba porque son los que importan.
random.seed(SEMILLA)
def split(lst, test_share, valid_share):
    lst = lst[:]; random.shuffle(lst)
    n_test, n_valid = round(len(lst) * test_share), round(len(lst) * valid_share)
    return lst[n_test + n_valid:], lst[n_test:n_test + n_valid], lst[:n_test]
parts = {"train": [], "valid": [], "test": []}
for group, shares in ((public, (0.1, 0.1)), (own, (0.25, 0.15))):
    tr, va, te = split(group, *shares)
    parts["train"] += tr; parts["valid"] += va; parts["test"] += te

for part, entries in parts.items():
    folder = DATASET / part; folder.mkdir(parents=True)
    images, annotations = [], []
    for i, (path, boxes) in enumerate(entries):
        name = f"{i:05d}{path.suffix.lower()}"
        shutil.copy(path, folder / name)
        w, h = Image.open(path).size
        images.append({"id": i, "file_name": name, "width": w, "height": h})
        for label, x, y, bw, bh in boxes:
            annotations.append({"id": len(annotations), "image_id": i, "category_id": CLASES.index(label),
                                "bbox": [x, y, bw, bh], "area": bw * bh, "iscrowd": 0})
    (folder / "_annotations.coco.json").write_text(json.dumps(
        {"images": images, "annotations": annotations, "categories": CATEGORIES}))
    print(part, len(images), "imágenes,", len(annotations), "frutos")

## 4. Entrenar

In [ ]:
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRLarge
Model = {"nano": RFDETRNano, "small": RFDETRSmall, "medium": RFDETRMedium, "large": RFDETRLarge}[TAMANO]
OUTPUT = ROOT / "entrenamiento"
model = Model()
model.train(dataset_dir=str(DATASET), epochs=EPOCAS, batch_size=8, grad_accum_steps=2, lr=1e-4,
            output_dir=str(OUTPUT), early_stopping=True, early_stopping_patience=8)
BEST = OUTPUT / "checkpoint_best_total.pth"
assert BEST.exists(), list(OUTPUT.iterdir())

## 5. Error de conteo en imágenes no vistas

Lo que importa para Atias no es solo detectar cajas, sino **contar bien los frutos maduros**. Se compara, imagen a imagen, cuántos maduros cuenta el modelo frente a los etiquetados a mano, y se elige el umbral de confianza que menos se equivoca.

In [ ]:
import numpy as np
model = Model(pretrain_weights=str(BEST))
print("Clases del modelo:", model.class_names)
test = json.loads((DATASET / "test" / "_annotations.coco.json").read_text())
truth = {img["id"]: 0 for img in test["images"]}
for a in test["annotations"]:
    if CLASES[a["category_id"]] in MADURAS:
        truth[a["image_id"]] += 1

def ripe_count(det):
    names = det.data.get("class_name")
    return sum(1 for n in names if n in MADURAS) if names is not None else 0

best = None
for threshold in (0.3, 0.4, 0.5, 0.6):
    real, counted = [], []
    for img in test["images"]:
        image = Image.open(DATASET / "test" / img["file_name"]).convert("RGB")
        real.append(truth[img["id"]]); counted.append(ripe_count(model.predict(image, threshold=threshold)))
    real, counted = np.array(real), np.array(counted)
    total_error = abs(counted.sum() - real.sum()) / max(real.sum(), 1) * 100
    per_image = np.abs(counted - real).sum() / max(real.sum(), 1) * 100
    print(f"umbral {threshold}: maduros reales {real.sum()}, contados {counted.sum()}, "
          f"error total {total_error:.1f} %, error imagen a imagen {per_image:.1f} %")
    if best is None or per_image < best[1]:
        best = (threshold, per_image, total_error)
UMBRAL, ERROR_IMAGEN, ERROR_TOTAL = best
print(f"\nUmbral elegido: {UMBRAL} · error imagen a imagen {ERROR_IMAGEN:.1f} %")
if MODO == "fruto_color":
    print("Modo fruto + color: es el error al contar frutos de cualquier madurez; la madurez la decide Atias por el color.")
print("Ojo: son fotos sueltas, no vídeo; el error real se mide con los tramos de control.")

## 6. Exportar a ONNX y comprobar

El servicio de análisis no usa PyTorch (pesa demasiado para un ordenador sin GPU): usa el modelo exportado a ONNX con el mismo preprocesado que `src/atias/vision/onnx_detector.py`. Esta celda comprueba que el ONNX da las mismas detecciones que el modelo original y deduce el orden de las clases.

In [ ]:
import onnxruntime as ort
export_dir = ROOT / "onnx"
shutil.rmtree(export_dir, ignore_errors=True)
model.export(output_dir=str(export_dir))
onnx_file = next(export_dir.glob("*.onnx"))
session = ort.InferenceSession(str(onnx_file), providers=["CPUExecutionProvider"])
print("Salidas:", [o.name for o in session.get_outputs()], "· entrada:", session.get_inputs()[0].shape)
RES = session.get_inputs()[0].shape[-1]

MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def onnx_detect(image, threshold):
    w, h = image.size
    x = (np.asarray(image.resize((RES, RES)), np.float32) / 255 - MEAN) / STD
    dets, logits = session.run(["dets", "labels"], {"input": x.transpose(2, 0, 1)[None]})
    probs = 1 / (1 + np.exp(-logits[0][:, :len(CLASES)]))
    ids, scores = probs.argmax(1), probs.max(1)
    keep = scores >= threshold
    cx, cy, bw, bh = dets[0][keep].T
    boxes = np.stack([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h], 1)
    return boxes, ids[keep]

def iou(a, b):
    x1, y1 = np.maximum(a[0], b[:, 0]), np.maximum(a[1], b[:, 1])
    x2, y2 = np.minimum(a[2], b[:, 2]), np.minimum(a[3], b[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]) - inter)

order, matched, total = {}, 0, 0
for img in test["images"][:15]:
    image = Image.open(DATASET / "test" / img["file_name"]).convert("RGB")
    ref = model.predict(image, threshold=UMBRAL)
    boxes, ids = onnx_detect(image, UMBRAL)
    for box, name in zip(ref.xyxy, ref.data["class_name"]):
        total += 1
        if len(boxes) == 0: continue
        overlap = iou(box, boxes)
        j = int(overlap.argmax())
        if overlap[j] > 0.7:
            matched += 1
            order.setdefault(int(ids[j]), set()).add(name)
print(f"Detecciones del original encontradas en ONNX: {matched}/{total}")
assert total == 0 or matched / total > 0.9, "El ONNX no coincide con el original: no usar este modelo"
assert all(len(v) == 1 for v in order.values()), f"Orden de clases ambiguo: {order}"
assert all(n in CLASES for v in order.values() for n in v), f"Clases inesperadas: {order}"
CLASES_ONNX = [next(iter(order[i])) if i in order else CLASES[i] for i in range(len(CLASES))]
print("Orden de clases en el ONNX:", CLASES_ONNX)

## 7. Empaquetar y descargar

In [ ]:
import datetime, time
from google.colab import files
name = f"rfdetr-{TAMANO}-{CULTIVO}-{datetime.date.today().isoformat()}"
package = ROOT / "paquete" / CULTIVO
shutil.rmtree(package.parent, ignore_errors=True); package.mkdir(parents=True)
shutil.copy(onnx_file, package / "modelo.onnx")
config = {
    "nombre": name, "cultivo": CULTIVO, "resolucion": int(RES), "clases": CLASES_ONNX,
    "equivalencias": EQUIVALENCIAS, "umbral": UMBRAL, "teselas": TESELAS, "solape": SOLAPE,
    "ancho_max": ANCHO_MAX,
    "evaluacion": {"imagenes_prueba": len(test["images"]), "error_imagen_pct": round(ERROR_IMAGEN, 1),
                   "error_total_pct": round(ERROR_TOTAL, 1)},
    "datos": LICENCIAS,
}
(package / "modelo.json").write_text(json.dumps(config, ensure_ascii=False, indent=2))
(package / "LICENCIAS.txt").write_text("RF-DETR: Apache 2.0 (Roboflow).\n" + "\n".join(LICENCIAS) + "\n")

# Velocidad en CPU, para saber cuánto tardará un vídeo en el ordenador de Atias.
cpu = ort.InferenceSession(str(package / "modelo.onnx"), providers=["CPUExecutionProvider"])
x = np.zeros((1, 3, RES, RES), np.float32); cpu.run(None, {"input": x})
start = time.time()
for _ in range(10): cpu.run(None, {"input": x})
per_tile = (time.time() - start) / 10
minutes = per_tile * TESELAS[0] * TESELAS[1] * 3 * 600 / 60
print(f"{per_tile:.2f} s por tesela en la CPU de Colab: un vídeo de 10 min tardaría unos {minutes:.0f} min")

zip_file = shutil.make_archive(str(ROOT / f"modelo-{CULTIVO}"), "zip", package.parent)
files.download(zip_file)
print("Descomprímelo dentro de la carpeta modelos/ de Atias: quedará modelos/" + CULTIVO + "/modelo.onnx")